# Neuro-Immune Biomarker, Molecular Profiling, and Histological Imaging Data Exploration with `mlcroissant`
This notebook guides users through loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- Title: Neuro-Immune Biomarker, Molecular Profiling, and Histological Imaging Data from Animal Models and Human Subjects with Chronic Rhinosinusitis Eosinophilic Inflammation
- Schema URL: https://sen.science/doi/10.71728/senscience.aehw-2kj2/fair2.json


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.aehw-2kj2/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset Name:', metadata.name)
print('Description:', metadata.description)
print('License:', metadata.license)
print('Date Published:', metadata.datePublished)

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Discover record sets in the dataset
record_sets = dataset.record_sets
if not record_sets:
    print('No record sets found in the dataset.')
else:
    print('Record sets available:')
    for rs in record_sets:
        print(f"- Name: {rs.name}\n  @id: {rs['@id']}\n  Fields:")
        for field in rs.fields:
            print(f"    - Field Name: {field.name} | @id: {field['@id']} | Data type: {getattr(field, 'dataType', 'Unknown')}")
    # For demonstration, select first available record set
    primary_record_set_id = record_sets[0]['@id'] if record_sets else None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare extraction from each record set
dataframes = {}
record_set_ids = []

# Discover and extract data from available record sets
for rs in dataset.record_sets:
    rs_id = rs['@id']
    record_set_ids.append(rs_id)
    print(f"Loading records from record set @id: {rs_id} ({rs.name})")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for {rs_id}")

# For demonstration, select first record set with records
selected_rs_id = None
for rs_id in record_set_ids:
    if rs_id in dataframes and not dataframes[rs_id].empty:
        selected_rs_id = rs_id
        break
if selected_rs_id:
    print(f"Selected record set for analysis: {selected_rs_id}")
    df = dataframes[selected_rs_id]
else:
    print("No record set with records found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Example EDA: Identify numeric and categorical fields using their @id
if selected_rs_id:
    # Find a numeric field by @id
    numeric_field_id = None
    group_field_id = None
    rs_obj = next((rs for rs in dataset.record_sets if rs['@id'] == selected_rs_id), None)

    for f in rs_obj.fields:
        dtype = getattr(f, 'dataType', '').lower()
        if dtype in ['float', 'integer', 'number', 'schema:float', 'schema:integer', 'schema:number']:
            numeric_field_id = f['@id']
            break

    # Find a groupable/categorical field by @id
    for f in rs_obj.fields:
        dtype = getattr(f, 'dataType', '').lower()
        if dtype in ['text', 'string', 'schema:text']:
            group_field_id = f['@id']
            break

    print(f"Numeric field @id: {numeric_field_id}")
    print(f"Group field @id: {group_field_id}")

    # Proceed only if numeric_field_id exists
    if numeric_field_id and numeric_field_id in df.columns:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric field found or not in columns.")
else:
    print("No record set selected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization example
if selected_rs_id and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter plot for numeric vs group field if available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated how to load a FAIR^2 Croissant dataset using the `mlcroissant` library.
- We explored available record sets and fields by their `@id`, and extracted data into DataFrames.
- Simple EDA steps and visualizations were applied to numeric and categorical fields using their `@id`s.
- You can extend this notebook to deeper analyses or machine learning workflows using the provided dataset structure.